# Shape Feature Space

Explores the 82-feature morphometric representation extracted from each retained mask.
Features span four descriptor families: geometric, radial boundary profile, Hu moments,
and Elliptic Fourier Descriptors (EFD).


## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path('../data')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

df = pd.read_csv(DATA_DIR / 'shape_analysis_results.csv')
print(f'{len(df)} masks, {df.shape[1]} features')


## Feature families overview

In [ ]:
families = {
    'Geometric (8)':         ['area_mu2','perimeter_mu','circularity','solidity','convexity','aspect_ratio','eccentricity','orientation_deg'],
    'Radial profile (7)':    ['radial_mean_mu','radial_std_mu','radial_min_mu','radial_max_mu','radial_cv','n_radial_peaks','asymmetry_index'],
    'Hu moments (7)':        [f'hu{i}' for i in range(1, 8)],
    'EFD coefficients (40)': [c for c in df.columns if c.startswith('efd_') and c[4] in 'abcd'],
    'EFD power (10)':        [f'efd_power_h{i}' for i in range(1, 11)],
    'EFD summary (1)':       ['efd_deviation'],
}
for name, cols in families.items():
    print(f'  {name}: {len(cols)} features')
print(f'  Total: {sum(len(v) for v in families.values())}')


## Key geometric feature distributions

In [ ]:
geo_features = ['area_mu2', 'circularity', 'eccentricity', 'aspect_ratio', 'solidity', 'convexity']
labels = ['Area (µm²)', 'Circularity', 'Eccentricity', 'Aspect ratio', 'Solidity', 'Convexity']

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, feat, label in zip(axes.flat, geo_features, labels):
    ax.hist(df[feat].dropna(), bins=35, color='#4C72B0', edgecolor='white', linewidth=0.3)
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(f'{label}\nmedian={df[feat].median():.3f}')
plt.suptitle('Geometric feature distributions (n=1,737)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


## Shape class frequencies

In [ ]:
counts = df['shape_class'].value_counts()
pct    = (counts / len(df) * 100).round(1)
freq   = pd.DataFrame({'Count': counts, 'Percent': pct})
print(freq.to_string())

fig, ax = plt.subplots(figsize=(8, 3.5))
colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860','#DA8BC3']
ax.barh(counts.index, counts.values, color=colors[:len(counts)])
for i, (n, p) in enumerate(zip(counts.values, pct.values)):
    ax.text(n + 5, i, f'{p}%', va='center', fontsize=10)
ax.set_xlabel('Count')
ax.set_title('Shape class distribution')
plt.tight_layout()
plt.show()


## EFD harmonic power spectrum

In [ ]:
hp_cols = [f'efd_power_h{i}' for i in range(1, 11)]
mean_power = df[hp_cols].mean()
cum_power  = mean_power.cumsum() / mean_power.sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.bar(range(1, 11), mean_power.values, color='steelblue')
ax1.set_xlabel('Harmonic'); ax1.set_ylabel('Mean power'); ax1.set_title('Mean harmonic power')

ax2.plot(range(1, 11), cum_power.values, 'o-', color='steelblue')
ax2.axhline(0.95, color='crimson', linestyle='--', label='95%')
ax2.axvline(3, color='grey', linestyle=':', alpha=0.7, label='K=3 (reconstruction target)')
ax2.set_xlabel('Harmonic'); ax2.set_ylabel('Cumulative power'); ax2.set_title('Cumulative power')
ax2.legend()
plt.tight_layout()
plt.show()
print('Cumulative power by harmonic:')
for n, c in enumerate(cum_power, 1):
    print(f'  H{n}: {c:.3f}')
